# 10. Faithfulness and Groundedness

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/evals/10_faithfulness_and_groundedness.ipynb)

Retrieval is only half the story. A system can fetch the right evidence and still answer badly. In this notebook you will score whether an answer is actually supported by the source material.

## 🎯 Learning Objectives

By the end of this notebook, you will:

- Understand **Learning goals**
- Understand **Why this notebook matters**
- Understand **Step 1 — Utility helpers for claim-level analysis**
- Understand **Step 2 — Rebuild a tiny local corpus and sentence-level passages**
- Understand **Step 3 — Generate one grounded answer and one ungrounded answer**


## Learning goals

- compare grounded and ungrounded answers on the same question
- split answers into claims and score support claim by claim
- measure quote alignment and unsupported-claim rate
- build a small faithfulness benchmark without using eval packages

## Why this notebook matters

A RAG answer can sound polished while quietly inventing details. Faithfulness evaluation asks a narrower question than overall quality: did the answer stay inside the evidence boundary?

In [ ]:
# --- Google Colab Setup ---
!pip install -q "transformers>=4.51.0" accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

import csv
import json
import math
import random
import statistics
import time
from pathlib import Path
from typing import Any, Dict, List

import faiss
import numpy as np
import torch
from google.colab import drive
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"
EMBEDDING_MODEL_NAME = "BAAI/bge-base-en-v1.5"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME, cache_folder=CACHE_DIR)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True, top_k=20):
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None,
        do_sample=do_sample,
        top_k=top_k,
        pad_token_id=tokenizer.pad_token_id,
    )
    generated_ids = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

def embed_texts(texts, batch_size=32):
    if isinstance(texts, str):
        texts = [texts]
    return embed_model.encode(
        list(texts),
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )

def build_faiss_index(texts):
    texts = list(texts)
    embeddings = embed_texts(texts).astype(np.float32)
    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    return {
        "texts": texts,
        "embeddings": embeddings,
        "index": index,
    }

def search_index(query, store, k=5):
    k = min(k, len(store["texts"]))
    if k <= 0:
        return []

    query_vector = embed_texts([query]).astype(np.float32)
    scores, indices = store["index"].search(query_vector, k)
    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        results.append(
            {
                "index": int(idx),
                "score": float(score),
                "text": store["texts"][idx],
            }
        )
    return results

print(f"✓ Model loaded: {MODEL_NAME}")
print(f"✓ Embedding model loaded: {EMBEDDING_MODEL_NAME}")
print(f"  GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

## Step 1 — Utility helpers for claim-level analysis

We will reuse the same tiny local corpus idea from Notebook 09, but now we will work at passage level so each answer claim can be compared with its best supporting evidence.

In [ ]:
import re

random.seed(11)

STOPWORDS = {
    "the", "a", "an", "and", "or", "to", "of", "in", "for", "on", "with", "by",
    "at", "from", "during", "into", "over", "under", "after", "before", "than",
    "what", "which", "where", "when", "how", "did", "does", "is", "are", "was",
    "were", "be", "been", "it", "its", "their", "that", "this", "these", "those"
}


def normalize(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    return re.sub(r"\s+", " ", text).strip()


def content_words(text):
    return [token for token in normalize(text).split() if token not in STOPWORDS]


def split_sentences(text):
    cleaned = text.replace("\n", " ").strip()
    parts = re.split(r"(?<=[.!?])\s+", cleaned)
    sentences = []
    for part in parts:
        for subpart in part.split(";"):
            subpart = subpart.strip()
            if subpart:
                sentences.append(subpart)
    return sentences


def extract_numbers(text):
    return set(re.findall(r"\d+(?:\.\d+)?", text))


def clip(text, width=80):
    text = str(text)
    return text if len(text) <= width else text[: width - 3] + "..."


def print_rows(rows, columns=None, limit=None):
    rows = list(rows)
    if limit is not None:
        rows = rows[:limit]
    if not rows:
        print("(no rows)")
        return
    if columns is None:
        columns = list(rows[0].keys())
    widths = {}
    for column in columns:
        widths[column] = len(str(column))
        for row in rows:
            widths[column] = max(widths[column], len(str(row.get(column, ""))))
    header = " | ".join(str(column).ljust(widths[column]) for column in columns)
    divider = "-+-".join("-" * widths[column] for column in columns)
    print(header)
    print(divider)
    for row in rows:
        print(" | ".join(str(row.get(column, "")).ljust(widths[column]) for column in columns))

## Step 2 — Rebuild a tiny local corpus and sentence-level passages

Sentence-level passages make support checking easier because we can compare each claim to a small evidence unit instead of a whole document.

In [ ]:
CORPUS = [
    {
        "doc_id": "D1",
        "title": "Harbor District Microgrid Pilot",
        "text": "The Harbor district launched a resilience pilot in 2024. It installed 6 megawatts of rooftop solar and a 20 megawatt-hour battery. During two grid outages, the microgrid kept the health clinic and water pumps online for 11 hours. Operators reported a 34 percent reduction in diesel generator use compared with the previous summer.",
    },
    {
        "doc_id": "D2",
        "title": "Cedar Wastewater Heat Recovery Memo",
        "text": "The Cedar treatment plant now captures heat from outgoing effluent. The recovered heat keeps digesters warm and supplies a nearby greenhouse loop. Plant managers reported an 18 percent reduction in natural gas consumption. The memo estimates a payback period of about 5.5 years.",
    },
    {
        "doc_id": "D3",
        "title": "Northwind Offshore Wind Siting Brief",
        "text": "The Northwind team shifted turbine placement 9 kilometers east to avoid a major bird migration corridor. The export cable shares an existing shipping channel for the first 14 kilometers. Environmental review says construction noise is the largest short-term risk. The project brief estimates annual output could power 210000 homes.",
    },
    {
        "doc_id": "D4",
        "title": "Library Retrofit Case Study",
        "text": "A central library replaced chillers with ground-source heat pumps and smart ventilation controls. Energy use intensity fell from 212 to 148 kilowatt-hours per square meter. Summer comfort complaints dropped by 40 percent. Total project cost was 4.2 million dollars with an 11-year payback.",
    },
    {
        "doc_id": "D5",
        "title": "Drought Response Handbook",
        "text": "Stage two restrictions begin when reservoir storage stays below 62 percent for 21 consecutive days. The handbook says leak repairs can save up to 12 percent of summer demand. Recharge basins should rest every seventh day to prevent compaction. Public messaging focuses on irrigation schedules and the repair hotline.",
    },
    {
        "doc_id": "D6",
        "title": "Transit Electrification Plan",
        "text": "Phase one purchases 18 battery buses by 2026. The full program targets 42 electric buses by 2028. Overnight depot charging serves most vehicles, while an on-route pantograph at Central Station handles the peak corridor. The plan forecasts 22 percent lower maintenance cost than diesel operations.",
    },
    {
        "doc_id": "D7",
        "title": "Estuary Flood Barrier Memo",
        "text": "The estuary barrier closes only during storm-surge warnings above 1.7 meters. Wetlands restoration is expected to delay peak flooding by 35 minutes. The annual maintenance budget is 8.4 million dollars. Fishing access is preserved through a side channel.",
    },
    {
        "doc_id": "D8",
        "title": "Data Center Water Reuse Project",
        "text": "A reclaim system now reuses 68 percent of cooling water at the municipal data center. The facility pairs with treated greywater from the wastewater plant. Summer potable water demand fell by 41 percent after deployment. Filtration membranes are replaced every 18 months.",
    },
]


def make_passages(corpus):
    passages = []
    for doc in corpus:
        for idx, sentence in enumerate(split_sentences(doc["text"]), start=1):
            passages.append(
                {
                    "passage_id": "{}-P{}".format(doc["doc_id"], idx),
                    "doc_id": doc["doc_id"],
                    "title": doc["title"],
                    "text": sentence,
                }
            )
    return passages


PASSAGES = make_passages(CORPUS)
PASSAGE_TEXTS = [passage["title"] + ". " + passage["text"] for passage in PASSAGES]
PASSAGE_STORE = build_faiss_index(PASSAGE_TEXTS)

print_rows(PASSAGES[:10], columns=["passage_id", "doc_id", "text"])

In [ ]:
QUESTION = "How did the Harbor district microgrid improve resilience, and what diesel savings were reported?"


def retrieve_context(question, k=4):
    results = search_index(question, PASSAGE_STORE, k=k)
    rows = []
    for result in results:
        passage = PASSAGES[result["index"]]
        rows.append(
            {
                "passage_id": passage["passage_id"],
                "doc_id": passage["doc_id"],
                "title": passage["title"],
                "score": round(result["score"], 4),
                "text": passage["text"],
            }
        )
    return rows


context_rows = retrieve_context(QUESTION, k=4)
print("Question:", QUESTION)
print_rows(context_rows, columns=["passage_id", "doc_id", "score", "text"])

## Step 3 — Generate one grounded answer and one ungrounded answer

A grounded answer sees retrieved context. An ungrounded answer gets only the question. We will score both outputs with the same support checker.

In [ ]:
def build_context_block(passages):
    blocks = []
    for passage in passages:
        blocks.append("[{0}] {1} ({2})".format(passage["passage_id"], passage["title"], passage["doc_id"]))
        blocks.append(passage["text"])
        blocks.append("")
    return "\n".join(blocks).strip()


context_block = build_context_block(context_rows)

grounded_prompt = """Use only the context below. If the context is insufficient, say so.
Question: {question}

Context:
{context}

Answer in 3 short sentences. Include one short direct quote from the context and cite the supporting document id like [D1].""".format(
    question=QUESTION,
    context=context_block,
)

ungrounded_prompt = """Answer the question from general intuition without using any external context.
Question: {question}

Answer in 3 short sentences.""".format(question=QUESTION)

grounded_answer = generate(grounded_prompt, max_new_tokens=180, temperature=0.0, do_sample=False, top_k=1)
ungrounded_answer = generate(ungrounded_prompt, max_new_tokens=180, temperature=0.0, do_sample=False, top_k=1)

print("Grounded answer:\n")
print(grounded_answer)
print("\nUngrounded answer:\n")
print(ungrounded_answer)

## Step 4 — Split answers into atomic claims

Faithfulness is easier to inspect when you score short claims rather than whole paragraphs. We will strip citations, split sentences, then compare each claim to nearby evidence.

In [ ]:
def strip_citations(text):
    return re.sub(r"\[[^\]]+\]", "", text).strip()


def answer_claims(answer):
    claims = []
    for sentence in split_sentences(answer):
        cleaned = strip_citations(sentence)
        if cleaned:
            claims.append(cleaned)
    return claims


print("Grounded claims:")
for claim in answer_claims(grounded_answer):
    print("-", claim)

print("\nUngrounded claims:")
for claim in answer_claims(ungrounded_answer):
    print("-", claim)

## Step 5 — Measure quote alignment

If an answer includes quoted text, that quote should appear in the source material. Quote alignment is narrow but useful: it catches fabricated quotations very quickly.

In [ ]:
def extract_quotes(answer):
    return [quote.strip() for quote in re.findall(r'"([^"]+)"', answer) if quote.strip()]


def quote_alignment_ratio(answer, passages):
    quotes = extract_quotes(answer)
    if not quotes:
        return 0.0
    searchable = [normalize(passage["text"]) for passage in passages]
    aligned = 0
    for quote in quotes:
        q = normalize(quote)
        if any(q in text for text in searchable):
            aligned += 1
    return aligned / len(quotes)


print("Grounded quotes:", extract_quotes(grounded_answer))
print("Grounded quote alignment:", round(quote_alignment_ratio(grounded_answer, PASSAGES), 3))
print("Ungrounded quote alignment:", round(quote_alignment_ratio(ungrounded_answer, PASSAGES), 3))

In [ ]:
def overlap_score(claim, evidence):
    claim_terms = set(content_words(claim))
    evidence_terms = set(content_words(evidence))
    if not claim_terms:
        return 0.0
    lexical = len(claim_terms & evidence_terms) / len(claim_terms)
    claim_numbers = extract_numbers(claim)
    if claim_numbers:
        numeric = len(claim_numbers & extract_numbers(evidence)) / len(claim_numbers)
    else:
        numeric = 0.0
    return 0.75 * lexical + 0.25 * numeric


def evidence_search(claim, k=4):
    results = search_index(claim, PASSAGE_STORE, k=k)
    rows = []
    for result in results:
        passage = PASSAGES[result["index"]]
        rows.append(
            {
                "passage_id": passage["passage_id"],
                "doc_id": passage["doc_id"],
                "dense_score": round(result["score"], 4),
                "support_score": round(overlap_score(claim, passage["text"]), 3),
                "text": passage["text"],
            }
        )
    return sorted(rows, key=lambda row: (row["support_score"], row["dense_score"]), reverse=True)


def score_answer_support(answer, threshold=0.55):
    claims = answer_claims(answer)
    diagnostics = []
    for claim in claims:
        candidates = evidence_search(claim, k=4)
        best = candidates[0] if candidates else {"passage_id": "", "doc_id": "", "support_score": 0.0, "text": ""}
        diagnostics.append(
            {
                "claim": clip(claim, 64),
                "support_score": best["support_score"],
                "supported": best["support_score"] >= threshold,
                "doc_id": best["doc_id"],
                "passage_id": best["passage_id"],
                "evidence": clip(best["text"], 72),
            }
        )
    if diagnostics:
        mean_support = statistics.mean(row["support_score"] for row in diagnostics)
        supported_ratio = statistics.mean(1.0 if row["supported"] else 0.0 for row in diagnostics)
    else:
        mean_support = 0.0
        supported_ratio = 0.0
    return {
        "claim_count": len(diagnostics),
        "supported_claim_ratio": round(supported_ratio, 3),
        "unsupported_claim_score": round(1.0 - supported_ratio, 3),
        "mean_best_support": round(mean_support, 3),
        "quote_alignment_ratio": round(quote_alignment_ratio(answer, PASSAGES), 3),
        "diagnostics": diagnostics,
    }

## Step 6 — Score grounded versus ungrounded answers

The answer-level summary below is still simple, but it is already useful. A grounded answer should usually show higher support and better quote alignment than an ungrounded one.

In [ ]:
comparison_rows = []
for label, answer in [("grounded", grounded_answer), ("ungrounded", ungrounded_answer)]:
    scored = score_answer_support(answer)
    comparison_rows.append(
        {
            "label": label,
            "claim_count": scored["claim_count"],
            "supported_claim_ratio": scored["supported_claim_ratio"],
            "unsupported_claim_score": scored["unsupported_claim_score"],
            "quote_alignment_ratio": scored["quote_alignment_ratio"],
            "mean_best_support": scored["mean_best_support"],
        }
    )

print_rows(
    comparison_rows,
    columns=[
        "label",
        "claim_count",
        "supported_claim_ratio",
        "unsupported_claim_score",
        "quote_alignment_ratio",
        "mean_best_support",
    ],
)

## Step 7 — Build a tiny manual faithfulness benchmark

Model outputs can vary, so it is helpful to keep a few fixed answers around for regression testing. These examples include grounded, partially grounded, and clearly hallucinated behavior.

In [ ]:
ANSWER_BANK = [
    {
        "label": "grounded_manual",
        "question": "How did the Harbor district microgrid improve resilience, and what diesel savings were reported?",
        "answer": "The Harbor pilot installed 6 megawatts of rooftop solar and a 20 megawatt-hour battery [D1]. During outages it kept the clinic and water pumps online for 11 hours and the report says diesel generator use fell by 34 percent, a \"reduction in diesel generator use\" [D1].",
    },
    {
        "label": "partly_grounded",
        "question": "How did the Harbor district microgrid improve resilience, and what diesel savings were reported?",
        "answer": "The Harbor pilot paired solar with a battery and kept essential services online for 11 hours [D1]. It also exported excess power to the neighboring port and cut diesel use by 60 percent [D1].",
    },
    {
        "label": "hallucinated",
        "question": "How did the Harbor district microgrid improve resilience, and what diesel savings were reported?",
        "answer": "The Harbor system relied on hydrogen fuel cells and delivered 72 hours of autonomous backup. Diesel generators were completely eliminated within one month of launch.",
    },
    {
        "label": "cross_doc_stitching",
        "question": "How did the Harbor district microgrid improve resilience, and what diesel savings were reported?",
        "answer": "The Harbor pilot kept the clinic online for 11 hours and used an on-route pantograph at Central Station to charge its fleet [D1, D6]. Diesel use fell by 34 percent [D1].",
    },
]

print_rows(
    [{"label": row["label"], "answer": clip(row["answer"], 92)} for row in ANSWER_BANK],
    columns=["label", "answer"],
)

In [ ]:
batch_rows = []
for sample in ANSWER_BANK:
    scored = score_answer_support(sample["answer"])
    batch_rows.append(
        {
            "label": sample["label"],
            "supported_claim_ratio": scored["supported_claim_ratio"],
            "unsupported_claim_score": scored["unsupported_claim_score"],
            "quote_alignment_ratio": scored["quote_alignment_ratio"],
            "mean_best_support": scored["mean_best_support"],
        }
    )

print_rows(
    batch_rows,
    columns=[
        "label",
        "supported_claim_ratio",
        "unsupported_claim_score",
        "quote_alignment_ratio",
        "mean_best_support",
    ],
)

## Step 8 — Inspect the weakest answer claim by claim

Answer-level numbers are useful for dashboards, but debugging still happens at claim level. That is where you discover whether the model invented a number, mixed documents, or quoted text that never existed.

In [ ]:
weakest = min(ANSWER_BANK, key=lambda row: score_answer_support(row["answer"])["supported_claim_ratio"])
weakest_score = score_answer_support(weakest["answer"])

print("Weakest sample:", weakest["label"])
print(weakest["answer"])
print()
print_rows(
    weakest_score["diagnostics"],
    columns=["claim", "supported", "support_score", "doc_id", "passage_id", "evidence"],
)

## Step 9 — Calibrate the support threshold

A support checker always contains a judgment call. If the threshold is too high, paraphrases look unsupported. If it is too low, hallucinations sneak through. Calibration is part of the work.

In [ ]:
threshold_rows = []
calibration_answer = ANSWER_BANK[1]["answer"]
for threshold in [0.45, 0.55, 0.65]:
    scored = score_answer_support(calibration_answer, threshold=threshold)
    threshold_rows.append(
        {
            "threshold": threshold,
            "supported_claim_ratio": scored["supported_claim_ratio"],
            "unsupported_claim_score": scored["unsupported_claim_score"],
            "mean_best_support": scored["mean_best_support"],
        }
    )

print("Calibration sample:", ANSWER_BANK[1]["label"])
print_rows(threshold_rows, columns=["threshold", "supported_claim_ratio", "unsupported_claim_score", "mean_best_support"])

## Recap

You now have a local faithfulness harness with claim extraction, quote alignment, support search, and unsupported-claim scoring. In the next notebook we will push one step further by checking whether answers cite the right sources completely and explicitly.

## 🏋️ Exercises

Put your understanding to the test:

**Exercise 1 — Explore:** design an eval dataset for a new use case. Document what changes and why.

**Exercise 2 — Build:** implement a custom metric and compare it to the one in this notebook. Compare your implementation to the one in this notebook.

**Exercise 3 — Extend:** run the evaluation on a different model and analyze the differences.

## 📚 References & Further Reading

- [Hugging Face Documentation](https://huggingface.co/docs)
- [LLM Course Resources](https://github.com/cyruslayo/castalia)
- Explore related notebooks in the evals/ directory